In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# project root
os.chdir('c:/dev/AnalystLab_Internship')

# Now use relative path from root
netflix_path = "Week_1/data/netflix_titles.csv"
retail_path = "Week_1/data/online_retail/OnlineRetail.csv"

netflix_df = pd.read_csv(netflix_path)
retail_df = pd.read_csv(retail_path, encoding='cp1252')

# Worked fine, so I commented out
# netflix_df.head(2)
# retail_df.head(2)


Netflix

In [ ]:
netflix_df['date_added'] = pd.to_datetime(
    netflix_df['date_added'].str.strip(),  # Remove spaces first
    errors='coerce'
)

# Then format if you want string format
netflix_df['date_added'] = netflix_df['date_added'].dt.strftime('%Y-%m-%d')

netflix_df.head(2)

In [ ]:
print("Netflix Dataset Shape:", netflix_df.shape)
print("\nMissing Values Analysis:")
missing_counts = netflix_df.isnull().sum()
missing_pct = (missing_counts / len(netflix_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False))

# Strategic handling based on business context
def clean_netflix_data(df):
    df_clean = df.copy()

    # Using 'Unknown' preserves the data for analysis without losing rows
    df_clean['director'] = df_clean['director'].fillna('Unknown')
    print("✓ Director: Filled 2,634 missing values with 'Unknown'")

    # 'Not Available' is more appropriate than 'Unknown' for cast
    df_clean['cast'] = df_clean['cast'].fillna('Not Available')
    print("✓ Cast: Filled 825 missing values with 'Not Available'")
    
    df_clean['country'] = df_clean['country'].fillna('Unknown')
    print(f"✓ Country: Filled 831 missing values with 'Unknown'")
    
    # Better to drop few rows than create incorrect dates
    df_clean = df_clean.dropna(subset=['date_added'])
    print("✓ Date Added: Dropped 10 rows with missing dates")
    
    # Very few missing, mode imputation is safe
    most_common_rating = df_clean['rating'].mode()[0]
    df_clean['rating'] = df_clean['rating'].fillna(most_common_rating)
    print(f"✓ Rating: Filled 4 missing values with '{most_common_rating}'")

    # Only 3 missing, safer to drop than impute incorrectly
    df_clean = df_clean.dropna(subset=['duration'])
    print("✓ Duration: Dropped 3 rows with missing duration")
    
    return df_clean

# Clean the data
netflix_df_clean = clean_netflix_data(netflix_df)

# Verify cleaning
print("\n" + "="*50)
print("CLEANING SUMMARY")
print("="*50)
print(f"Original shape: {netflix_df.shape}")
print(f"Cleaned shape: {netflix_df_clean.shape}")
print(f"Total rows removed: {netflix_df.shape[0] - netflix_df_clean.shape[0]}")
print(f"Remaining missing values: {netflix_df_clean.isnull().sum().sum()}")

# Quality check - show sample of cleaned columns
print("\nSample of cleaned data:")
print(netflix_df_clean[['director', 'cast', 'country', 'rating']].head(3))

In [ ]:
# Clean and convert date_added to datetime
netflix_df['date_added'] = pd.to_datetime(netflix_df['date_added'].str.strip(), errors='coerce')
netflix_df['year_added'] = netflix_df['date_added'].dt.year


1. Movies vs TV Shows distribution


In [ ]:
type_counts = netflix_df['type'].value_counts()
print(type_counts)

import matplotlib.pyplot as plt
type_counts.plot(kind='pie', autopct='%1.1f%%', figsize=(6,6))
plt.title('Movies vs TV Shows')
plt.ylabel('')
plt.show()

2. Content added by year


In [ ]:
content_by_year = netflix_df['year_added'].value_counts().sort_index()
print(content_by_year)

content_by_year.plot(kind='line', marker='o', figsize=(10,5))
plt.title('Content Added to Netflix by Year')
plt.xlabel('Year')
plt.ylabel('Number of Titles Added')
plt.show()

3. Top content-producing countries


In [ ]:
top_countries = (
    netflix_df['country']
    .dropna()
    .str.split(', ')
    .explode()
    .value_counts()
    .head(10)
)
print(top_countries)

top_countries.plot(kind='barh', figsize=(8,5), color='crimson')
plt.title('Top 10 Content-Producing Countries')
plt.xlabel('Number of Titles')
plt.gca().invert_yaxis()
plt.show()

4. Most common ratings


In [ ]:
rating_counts = netflix_df['rating'].value_counts().head(10)
print(rating_counts)

rating_counts.plot(kind='bar', figsize=(8,5), color='purple')
plt.title('Most Common Content Ratings')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

5. Most common genres/categories


In [ ]:
top_genres = (
    netflix_df['listed_in']
    .dropna()
    .str.split(', ')
    .explode()
    .value_counts()
    .head(10)
)
print(top_genres)

top_genres.plot(kind='barh', figsize=(8,5), color='teal')
plt.title('Top 10 Genres/Categories on Netflix')
plt.xlabel('Number of Titles')
plt.gca().invert_yaxis()
plt.show()

Retail

In [ ]:
# ============================================
# RETAIL DATA CLEANING PIPELINE
# ============================================
print(f"Original shape: {retail_df.shape}")

retail_df_clean = retail_df.copy()

# --- 1. Missing Values ---
missing_percentages = (retail_df.isnull().sum() / len(retail_df)) * 100

# CustomerID: two versions for different use cases
retail_customer_analytics = retail_df_clean.dropna(subset=['CustomerID']).copy()   # for customer analytics
retail_sales_analytics = retail_df_clean.copy()
retail_sales_analytics['CustomerID'] = retail_sales_analytics['CustomerID'].fillna(0)  # for sales analytics

# Description: fill placeholder (only ~0.27% missing)
retail_df_clean['Description'] = retail_df_clean['Description'].fillna('Unknown')

# --- 2. Duplicates ---
duplicate_count = retail_df_clean.duplicated().sum()
retail_df_clean = retail_df_clean.drop_duplicates()

# --- 3. Standardization ---
# Dates
retail_df_clean['InvoiceDate'] = pd.to_datetime(
    retail_df_clean['InvoiceDate'], format='%m/%d/%Y %H:%M', errors='coerce'
)
retail_df_clean['InvoiceYear'] = retail_df_clean['InvoiceDate'].dt.year
retail_df_clean['InvoiceMonth'] = retail_df_clean['InvoiceDate'].dt.month
retail_df_clean['InvoiceDay'] = retail_df_clean['InvoiceDate'].dt.day
retail_df_clean['InvoiceHour'] = retail_df_clean['InvoiceDate'].dt.hour

# Text formatting
retail_df_clean['Description'] = retail_df_clean['Description'].str.strip().str.upper()
retail_df_clean['Country'] = retail_df_clean['Country'].str.strip().str.title()
retail_df_clean['StockCode'] = retail_df_clean['StockCode'].astype(str).str.strip()
retail_df_clean['InvoiceNo'] = retail_df_clean['InvoiceNo'].astype(str).str.strip()

# Column names -> snake_case
retail_df_clean.rename(columns={
    'InvoiceNo': 'invoice_no', 'StockCode': 'stock_code', 'Description': 'description',
    'Quantity': 'quantity', 'InvoiceDate': 'invoice_date', 'UnitPrice': 'unit_price',
    'CustomerID': 'customer_id', 'Country': 'country'
}, inplace=True)

# Data types
retail_df_clean['customer_id'] = retail_df_clean['customer_id'].astype('float64')
retail_df_clean['quantity'] = retail_df_clean['quantity'].astype('int64')
retail_df_clean['unit_price'] = retail_df_clean['unit_price'].astype('float64')

categorical_cols = ['stock_code', 'description', 'country']
for col in categorical_cols:
    retail_df_clean[col] = retail_df_clean[col].astype('category')

# --- 4. Validation ---
negative_quantity = retail_df_clean[retail_df_clean['quantity'] < 0]        # valid returns, kept
zero_quantity = retail_df_clean[retail_df_clean['quantity'] == 0]
retail_df_clean = retail_df_clean[retail_df_clean['quantity'] != 0]         # removed

invalid_price = retail_df_clean[retail_df_clean['unit_price'] <= 0]
retail_df_clean = retail_df_clean[retail_df_clean['unit_price'] > 0]        # removed

price_99 = retail_df_clean['unit_price'].quantile(0.99)
high_prices = retail_df_clean[retail_df_clean['unit_price'] > price_99]     # flagged only

retail_df_clean['invoice_no'] = retail_df_clean['invoice_no'].astype(str)
cancelled_invoices = retail_df_clean[retail_df_clean['invoice_no'].str.startswith('C')]

missing_stock = retail_df_clean[retail_df_clean['stock_code'].isna()]
retail_df_clean = retail_df_clean.dropna(subset=['stock_code'])             # removed

# --- 5. Calculated Columns ---
retail_df_clean['total_price'] = retail_df_clean['quantity'] * retail_df_clean['unit_price']
retail_df_clean['is_return'] = retail_df_clean['quantity'] < 0
retail_df_clean['is_cancelled'] = retail_df_clean['invoice_no'].str.startswith('C')

# --- 6. Summary ---
rows_removed = 541909 - len(retail_df_clean)
print(f"Final shape: {retail_df_clean.shape} | Rows removed: {rows_removed:,} "
      f"| Kept: {(len(retail_df_clean)/541909)*100:.2f}%")
print(f"Memory: {retail_df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# --- 7. Analysis Versions ---
customer_analytics = retail_df_clean[retail_df_clean['customer_id'] > 0].copy()   # RFM, segmentation, CLV
sales_analytics = retail_df_clean.copy()                                          # sales trends, revenue
returns_analytics = retail_df_clean[retail_df_clean['is_return']].copy()          # return rate analysis
revenue_analytics = retail_df_clean[~retail_df_clean['is_cancelled']].copy()      # accurate revenue calc

# --- 8. Check Output ---
print(retail_df_clean[['invoice_no', 'stock_code', 'description', 'quantity',
                        'unit_price', 'customer_id', 'country', 'invoice_date',
                        'total_price', 'is_return', 'is_cancelled']].head())
print(retail_df_clean.dtypes)
print(retail_df_clean.isnull().sum()[retail_df_clean.isnull().sum() > 0])

In [ ]:
# ['Quantity', 'UnitPrice']
# Above are the numerical columns in the retail columns
mean = retail_df['Quantity'].mean()
median = retail_df['Quantity'].median()
std = retail_df['Quantity'].std()
min = retail_df['Quantity'].min()
max = retail_df['Quantity'].max()

print(f"""A summary for Quantity column: 
      1. Mean = {mean},
      2. Median = {median},
      3. std = {std},
      4. Min = {min}
      5. Max = {max}""")

mean = retail_df['UnitPrice'].mean()
median = retail_df['UnitPrice'].median()
std = retail_df['UnitPrice'].std()
min = retail_df['UnitPrice'].min()
max = retail_df['UnitPrice'].max()

print(f"""A summary for UnitPrice column: 
      1. Mean = {mean},
      2. Median = {median},
      3. std = {std},
      4. Min = {min}
      5. Max = {max}""")

In [ ]:
# Convert InvoiceDate to datetime
retail_df['InvoiceDate'] = pd.to_datetime(retail_df['InvoiceDate'])

# Create a Revenue column (Quantity * UnitPrice)
retail_df['Revenue'] = retail_df['Quantity'] * retail_df['UnitPrice']

# Optional but recommended: drop cancelled orders (InvoiceNo starting with 'C')
# and negative/zero quantities which are usually returns
retail_df = retail_df[~retail_df['InvoiceNo'].astype(str).str.startswith('C')]
retail_df = retail_df[retail_df['Quantity'] > 0]

1. Top-selling products (by quantity sold)


In [ ]:
top_products = retail_df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_products)

import matplotlib.pyplot as plt
top_products.plot(kind='barh', figsize=(8,5))
plt.title('Top 10 Products by Quantity Sold')
plt.xlabel('Quantity Sold')
plt.gca().invert_yaxis()
plt.show()

2. Highest revenue-generating countries


In [ ]:
top_countries = retail_df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
print(top_countries)

top_countries.plot(kind='bar', figsize=(8,5), color='green')
plt.title('Top 10 Countries by Revenue')
plt.ylabel('Revenue')
plt.show()

3. Monthly sales trends


In [ ]:
retail_df['Month'] = retail_df['InvoiceDate'].dt.to_period('M')
monthly_sales = retail_df.groupby('Month')['Revenue'].sum()

monthly_sales.plot(kind='line', marker='o', figsize=(10,5))
plt.title('Monthly Sales Trend')
plt.ylabel('Revenue')
plt.xlabel('Month')
plt.xticks(rotation=45)
plt.show()

4. Most purchased products (by number of orders/transactions, not just quantity)


In [ ]:
most_purchased = retail_df.groupby('Description')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)
print(most_purchased)

most_purchased.plot(kind='barh', figsize=(8,5), color='orange')
plt.title('Most Purchased Products (by number of orders)')
plt.xlabel('Number of Orders')
plt.gca().invert_yaxis()
plt.show()

5. Customer purchasing behavior


In [ ]:
# Total spend per customer
customer_spend = retail_df.groupby('CustomerID')['Revenue'].sum()

# Distribution of customer spend
customer_spend.plot(kind='hist', bins=50, figsize=(8,5))
plt.title('Distribution of Customer Spend')
plt.xlabel('Total Revenue per Customer')
plt.show()

# Distribution of order frequency (how many invoices per customer)
customer_orders = retail_df.groupby('CustomerID')['InvoiceNo'].nunique()
customer_orders.plot(kind='box', figsize=(5,5))
plt.title('Order Frequency per Customer (Box Plot)')
plt.show()